# Module 07.1 — I/O Behavior Analysis

Caracteriza y diagnostica el comportamiento de E/S a partir de las métricas agregadas de Darshan Parser.  
El notebook selecciona la campaña y delega el cálculo al módulo `07_1_io_behavior.py`.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from importlib.util import spec_from_file_location, module_from_spec

MODULE_DIR = Path("./Module").resolve()
MODULE07_1_FILE = MODULE_DIR / "07_1_io_behavior.py"

if not MODULE07_1_FILE.exists():
    raise FileNotFoundError(f"No se encontró Module 06: {MODULE07_1_FILE}")

spec = spec_from_file_location("deeptuneio_module07", MODULE07_1_FILE)
module07 = module_from_spec(spec)
sys.modules[spec.name] = module07
spec.loader.exec_module(module07)

print("Module 07:", MODULE07_1_FILE)
print("Version  :", module07.VERSION)


Module 07: F:\NoteBooks\CORA_DR_UAB\Tesis\Scripts\DeepTuneIO\07_Analysis\07_1_IO_Behavior\Module\07_1_io_behavior.py
Version  : 2.6.0


In [2]:
# ==============================================================================
# 1. Selección de campaña
# ============================================================================== 
CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": [
                "lustre_1ost",
                "lustre_2ost",
                "lustre_4ost",
                "nfs_SnGPU_e1",
            ],
        },
    },
    "DLIOv1": {
        "HDF5_64bs": {"scenarios": ["lustre_ost"]},
        "NPZ_64bs": {"scenarios": ["lustre_ost"]},
        "TFRecord_256kts_64bs": {"scenarios": ["lustre_ost"]},
        "TFRecord_1mts_64bs": {"scenarios": ["lustre_ost"]},
    },
}

APPLICATION_POS = 1
DATASET_POS = 1
SCENARIO_POS = 4

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
SCENARIO = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"][SCENARIO_POS - 1]

print("Application   :", APPLICATION)
print("Dataset group :", DATASET_GROUP)
print("Scenario      :", SCENARIO)


Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : nfs_SnGPU_e1


In [3]:
# ==============================================================================
# 2. Raíz de la campaña y rutas
# ============================================================================== 

PROJECT_ROOT = Path(r"D:\DL_Experiments\CORA_DR_UAB").resolve()
ARTICLE_ID = "Tesis"

ARTICLE_ROOT = PROJECT_ROOT / ARTICLE_ID
CAMPAIGN_ROOT = ARTICLE_ROOT / "Data" / APPLICATION / DATASET_GROUP / SCENARIO

print("Project root  :", PROJECT_ROOT)
print("Article root  :", ARTICLE_ROOT)
print("Campaign root :", CAMPAIGN_ROOT)
print("Exists        :", CAMPAIGN_ROOT.exists())


Project root  : D:\DL_Experiments\CORA_DR_UAB
Article root  : D:\DL_Experiments\CORA_DR_UAB\Tesis
Campaign root : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
Exists        : True


In [4]:

EVENT = "Tesis"

RESULTS_DIR = CAMPAIGN_ROOT / "results" 
INPUT_CSV = RESULTS_DIR / "Parser" / f"4b_Analisis_IO_{APPLICATION}_Parser_{SCENARIO}_{DATASET_GROUP}_{EVENT}_dep.csv"
OUTPUT_DIR = RESULTS_DIR / "Analysis" / "IO_Behavior"


print("Campaign root :", CAMPAIGN_ROOT)
print("Input CSV     :", INPUT_CSV)
print("Output dir    :", OUTPUT_DIR)


Campaign root : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
Input CSV     : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Parser\4b_Analisis_IO_DeepGalaxy_Parser_nfs_SnGPU_e1_DG_bw512_f3c5x64_Tesis_dep.csv
Output dir    : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Analysis\IO_Behavior


In [5]:
# ==============================================================================
# 3. Parámetros de análisis
# ============================================================================== 
# Referencia manual/fallback para el tamaño físico del dataset.
# Debe ajustarse cuando se seleccione otro dataset.
DATASET_SIZE_BYTES = 2038916648

# Tolerancia relativa para validar fuentes automáticas contra el valor manual.
DATASET_SIZE_TOLERANCE = 0.01  # 1 %

# Opcionales: se usan únicamente para validar el valor manual si están disponibles.
DATASET_PATH = None
DARSHAN_DATASET_SIZE_BYTES = None

# None = detección automática desde Process_IO.
# baseline -> mínimo número de procesos disponible
# extremos -> mínimo y máximo número de procesos disponibles
BASELINE_PROCESS = None
EXTREME_PROCESSES = None

# Configuraciones de procesos que se excluirán del análisis.
# La exclusión se aplica antes de calcular baseline, extremos, rho_rel y Delta.
EXCLUDE_PROCESSES = [16, 48]  # Ejemplo: [48, 128]

print("Dataset size manual :", DATASET_SIZE_BYTES)
print("Baseline             :", BASELINE_PROCESS if BASELINE_PROCESS is not None else "AUTO (min)")
print("Extreme processes    :", EXTREME_PROCESSES if EXTREME_PROCESSES is not None else "AUTO (min/max)")
print("Excluded processes   :", EXCLUDE_PROCESSES if EXCLUDE_PROCESSES else "None")


Dataset size manual : 2038916648
Baseline             : AUTO (min)
Extreme processes    : AUTO (min/max)
Excluded processes   : [16, 48]


In [6]:
# ==============================================================================
# 5. Construcción de argumentos y ejecución
# ============================================================================== 
args = [
    "--input-csv", str(INPUT_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--event", EVENT,
    "--dataset-size-bytes", str(DATASET_SIZE_BYTES),
    "--dataset-size-tolerance", str(DATASET_SIZE_TOLERANCE),
]

if DATASET_PATH is not None:
    args += ["--dataset-path", str(DATASET_PATH)]

if DARSHAN_DATASET_SIZE_BYTES is not None:
    args += ["--darshan-dataset-size-bytes", str(DARSHAN_DATASET_SIZE_BYTES)]

if BASELINE_PROCESS is not None:
    args += ["--baseline-process", str(BASELINE_PROCESS)]

if EXTREME_PROCESSES is not None:
    args += ["--extreme-processes", ",".join(str(v) for v in EXTREME_PROCESSES)]

if EXCLUDE_PROCESSES:
    args += ["--exclude-processes", ",".join(str(v) for v in EXCLUDE_PROCESSES)]

return_code = module07.main(args)
print("Return code:", return_code)


DeepTuneIO Analyzer — Module 07.1: I/O Behavior Analysis
Input CSV          : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Parser\4b_Analisis_IO_DeepGalaxy_Parser_nfs_SnGPU_e1_DG_bw512_f3c5x64_Tesis_dep.csv
Output directory   : D:\DL_Experiments\CORA_DR_UAB\Tesis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Analysis\IO_Behavior
Application        : DeepGalaxy
Dataset group      : DG_bw512_f3c5x64
Scenario           : nfs_SnGPU_e1
Event              : Tesis
Separator          : ','
Rows after filters : 17
Excluded processes : [16, 48]
Columns             : 66
Process selection by Access_Mode:
  0: available=[4, 8, 32, 64], baseline=4 (automatic_minimum), extremes=[4, 64] (automatic_min_max)
  1: available=[4, 8, 32, 64], baseline=4 (automatic_minimum), extremes=[4, 64] (automatic_min_max)
Dataset size manual : 2038916648
Dataset size FS     : None
Dataset size Darshan: 2038916648 (parser_POSIX_MAX_BYTE_READ)
Dataset size used   : 203

In [7]:
# ==============================================================================
# 6. Inspección de resultados
# ============================================================================== 
stem = f"{APPLICATION}_{DATASET_GROUP}_{SCENARIO}_{EVENT}"
OUTPUT_CSV = OUTPUT_DIR / f"07_1_io_behavior_{stem}.csv"
EXTREME_CSV = OUTPUT_DIR / f"07_1_io_behavior_extreme_cases_{stem}.csv"

if OUTPUT_CSV.exists():
    df_behavior = pd.read_csv(OUTPUT_CSV)
    display(df_behavior)
else:
    print("Output not found:", OUTPUT_CSV)

if EXTREME_CSV.exists():
    df_extremes = pd.read_csv(EXTREME_CSV)
    display(df_extremes)


,Access_Mode,Access_Mode_Label,Process_IO,n_runs,POSIX_READS_mean,POSIX_READS_std,POSIX_BYTES_READ_mean,POSIX_BYTES_READ_std,Sequential_Reads_pct_mean,Sequential_Reads_pct_std,...,Dataset_Size_Source,Dataset_Size_Validated,Dataset_Size_Filesystem_Relative_Error,Dataset_Size_Darshan_Relative_Error,Dataset_Size_Darshan_Origin,Rows_Before_Exclusions,Rows_After_Exclusions,Excluded_Processes,Available_Processes_Before,Available_Processes_After
0,0,shared,4,2,5.008860e+05,0.000000,2.041644e+09,0.000000e+00,97.584081,0.000000,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
1,0,shared,8,3,5.081040e+05,0.000000,2.044486e+09,0.000000e+00,97.048242,0.000000,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
2,0,shared,32,3,5.824620e+05,0.000000,2.187401e+09,0.000000e+00,94.265892,0.000000,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
3,0,shared,64,2,6.729900e+05,0.000000,2.340588e+09,0.000000e+00,91.366291,0.000000,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
4,1,shared_reload_shuffle,4,3,3.952389e+06,3706.654332,1.629571e+10,1.530008e+07,98.064108,0.000091,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
5,1,shared_reload_shuffle,8,2,7.869596e+06,25020.973452,3.244589e+10,1.034305e+08,98.063751,0.000172,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
6,1,shared_reload_shuffle,32,1,2.804532e+07,NaN,1.155689e+11,NaN,98.054970,NaN,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
7,1,shared_reload_shuffle,64,1,4.092304e+07,NaN,1.685450e+11,NaN,98.023253,NaN,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"


,Access_Mode,Access_Mode_Label,Process_IO,n_runs,POSIX_READS_mean,POSIX_READS_std,POSIX_BYTES_READ_mean,POSIX_BYTES_READ_std,Sequential_Reads_pct_mean,Sequential_Reads_pct_std,...,Dataset_Size_Source,Dataset_Size_Validated,Dataset_Size_Filesystem_Relative_Error,Dataset_Size_Darshan_Relative_Error,Dataset_Size_Darshan_Origin,Rows_Before_Exclusions,Rows_After_Exclusions,Excluded_Processes,Available_Processes_Before,Available_Processes_After
0,0,shared,4,2,5.008860e+05,0.000000,2.041644e+09,0.000000e+00,97.584081,0.000000,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
1,0,shared,64,2,6.729900e+05,0.000000,2.340588e+09,0.000000e+00,91.366291,0.000000,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
2,1,shared_reload_shuffle,4,3,3.952389e+06,3706.654332,1.629571e+10,1.530008e+07,98.064108,0.000091,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
3,1,shared_reload_shuffle,64,1,4.092304e+07,NaN,1.685450e+11,NaN,98.023253,NaN,...,darshan_validated,True,NaN,0.0,parser_POSIX_MAX_BYTE_READ,25,17,"16,48","4,8,16,32,48,64","4,8,32,64"
